In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode

# 1. 스파크 세션(지휘관) 생성
# master("local[*]")는 내 컴퓨터의 사용 가능한 모든 CPU 코어를 일꾼으로 쓰겠다는 뜻입니다.
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Spark_Total_Master") \
    .getOrCreate()

# 2. 하위 호환을 위한 SparkContext도 세션에서 안전하게 추출해 둡니다.
sc = spark.sparkContext

print("💡 1단계 완료: 스파크 지휘관이 정상적으로 깨어났습니다!")
print(f"현재 접속 유저: {sc.sparkUser()}")

💡 1단계 완료: 스파크 지휘관이 정상적으로 깨어났습니다!
현재 접속 유저: jovyan


In [2]:
# 1. 파이썬 기본 리스트로 가상 데이터 준비
raw_sentences = [
    "data processing with spark",
    "spark is faster than hadoop",
    "pyspark data processing framework"
]

# 2. parallelize: 일반 리스트를 일꾼들에게 조각내어 분산 배치 (RDD 변환)
text_rdd = sc.parallelize(raw_sentences)

# 3. Map 단계: 문장을 단어로 쪼개고(flatMap), 단어마다 꼬리표 1을 붙임(map -> emit(w,1))
words_rdd = text_rdd.flatMap(lambda line: line.split(" "))
pairs_rdd = words_rdd.map(lambda word: (word, 1))

# 4. Reduce 단계: 같은 단어끼리 네트워크로 모은 뒤(Shuffle), 꼬리표 숫자들을 다 더함
# (이때까지는 계획만 짜여있고, 실제 연산은 일어나지 않는 '지연 연산' 상태입니다)
word_counts_rdd = pairs_rdd.reduceByKey(lambda a, b: a + b)

# 5. Action 명령어: collect()를 만나는 순간 일꾼들이 진짜 연산을 해서 결과를 지휘관에게 보냅니다.
print("📊 RDD 워드카운트 결과:")
print(word_counts_rdd.collect())

📊 RDD 워드카운트 결과:
[('hadoop', 1), ('is', 1), ('processing', 2), ('spark', 2), ('faster', 1), ('pyspark', 1), ('with', 1), ('framework', 1), ('than', 1), ('data', 2)]


In [3]:
# 1. 2단계 실습을 위한 새로운 직원 데이터셋 준비 (이름, 부서, 연봉)
employee_data = [
    ("Alice", "개발팀", 4500),
    ("Bob", "개발팀", 6500),
    ("Charlie", "디자인팀", 5000),
    ("David", "마케팅팀", 7000),
    ("Eve", "개발팀", 4000)
]
columns = ["Name", "Department", "Salary"]

# 2. 데이터프레임 생성 (이 단계 역시 계획만 수립됨)
df = spark.createDataFrame(employee_data, schema=columns)

# 3. 가공 및 필터링 계획 세우기
# 연봉 10% 인상 컬럼을 만들고, 연봉이 5000 이상인 사람만 필터링
processed_df = df.withColumn("New_Salary", col("Salary") * 1.1) \
                 .filter(col("New_Salary") >= 5000)

# 4. Action 명령어: .show()를 만나는 순간 진짜 연산이 돌아가며 표가 출력됩니다.
print("📊 데이터프레임 가공 결과 (.show() 실행):")
processed_df.show()


📊 데이터프레임 가공 결과 (.show() 실행):
+-------+----------+------+-----------------+
|   Name|Department|Salary|       New_Salary|
+-------+----------+------+-----------------+
|    Bob|    개발팀|  6500|7150.000000000001|
|Charlie|  디자인팀|  5000|           5500.0|
|  David|  마케팅팀|  7000|7700.000000000001|
+-------+----------+------+-----------------+



In [4]:
# 1. 데이터프레임을 'emp_table'이라는 가상 SQL 테이블(뷰)로 등록
df.createOrReplaceTempView("emp_table")

# 2. 순수 ANSI SQL 문법으로 대용량 데이터 조회 쿼리 작성
# 부서별로 그룹화(GROUP BY)하여 총 연봉 합계와 평균 연봉을 계산
sql_query = """
    SELECT Department, 
           SUM(Salary) as Total_Salary, 
           AVG(Salary) as Avg_Salary
    FROM emp_table
    GROUP BY Department
    ORDER BY Total_Salary DESC
"""

# 3. 쿼리 실행 계획 제출
sql_result_df = spark.sql(sql_query)

# 4. Action 명령어: 최종 결과를 화면에 출력
print("📊 Spark SQL 조회 결과:")
sql_result_df.show()


📊 Spark SQL 조회 결과:
+----------+------------+----------+
|Department|Total_Salary|Avg_Salary|
+----------+------------+----------+
|    개발팀|       15000|    5000.0|
|  마케팅팀|        7000|    7000.0|
|  디자인팀|        5000|    5000.0|
+----------+------------+----------+



1. 스파크 성능의 최대 적: 셔플링 (Shuffling)앞서 우리가 맵리듀스 구조를 이야기할 때, 같은 단어들을 한곳으로 모으기 위해 데이터 대이동이 일어난다고 했습니다. 이를 셔플링이라고 합니다.왜 셔플링이 느릴까요?정의: 여러 컴퓨터(코어)에 흩어져 있는 데이터를 특정 기준(Key)으로 묶기 위해 네트워크를 통해 서로 데이터를 주고받는 현상입니다.문제점: 데이터가 랜선(네트워크)을 타고 이동하고, 하드디스크에 임시로 저장되는 과정에서 엄청난 병목(지연)이 발생합니다.연산 종류:셔플이 안 일어나는 착한 연산: map, filter (각자 자기 자리의 데이터만 보면 됨)셔플을 일으키는 무거운 연산: reduceByKey, groupBy, join (남의 자리에 있는 데이터와 합쳐야 함)💡 최적화 핵심 1: 실무에서 코드를 짜실 때는 "어떻게 하면 셔플(groupBy, join 등)을 최소한으로 일어나게 할까?"를 늘 고민해야 합니다.

2. 똑같은 계산 두 번 안 하기: 캐싱 (Caching)스파크는 지연 연산(Lazy Evaluation)을 한다고 배웠습니다. 이 특성 때문에 초보자들이 가장 많이 하는 실수가 있습니다.초보자의 실수 시나리오10TB짜리 거대한 로그 데이터를 불러와서 깨끗하게 정제했습니다. (refined_df)정제된 데이터로 A 분석 결과를 뽑아봅니다. ➡️ refined_df.groupBy(...).count().show() (Action!)똑같은 정제 데이터로 B 분석 결과를 뽑아봅니다. ➡️ refined_df.filter(...).show() (Action!)스파크의 억울한 행동스파크는 바보같이 2번과 3번 명령을 만날 때마다, 처음 10TB 파일을 읽는 것부터 정제하는 과정까지의 전체 계획을 매번 처음부터 다시 실행합니다. 메모리에 정제된 데이터를 저장해 두지 않았기 때문입니다.🛠️ 해결책: .cache()이때 "이 정제된 데이터는 앞으로 계속 쓸 거니까 일꾼들 메모리에 딱 붙여놔!"라고 명령하는 것이 바로 캐싱입니다.
# 정제된 데이터프레임을 메모리에 고정하라는 명령어 (Transformation)
refined_df.cache() 

# 이제 아래 Action들을 실행할 때, 처음 딱 한 번만 계산하고 
# 다음부터는 메모리에서 바로 꺼내 쓰기 때문에 수십 배 빨라집니다.
refined_df.groupBy(...).count().show()
refined_df.filter(...).show()
